In [1]:
# 모듈 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import classification_report

In [2]:
# 데이터 로딩
df = pd.read_csv('/content/drive/MyDrive/KDThome/santander-customer-transaction-prediction/train.csv')

In [3]:
df_1 = df.drop('ID_code', axis=1, inplace=False)

In [4]:
# 데이터 분리
x_data = df_1.drop('target', axis=1).values
t_data = df_1['target'].values

# train/test split
x_train, x_test, t_train, t_test = train_test_split(
    x_data, t_data,
    test_size=0.2,
    stratify=t_data,
)

In [5]:
# 3. StandardScaler 적용
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# 오버샘플링
sm = SMOTE(random_state=42)
x_train_sm, t_train_sm = sm.fit_resample(x_train_scaled, t_train)

In [7]:
# 모델 학습
model = XGBClassifier(
        n_estimators=700,
        max_depth=3,
        learning_rate=0.2,
)
model.fit(x_train_sm, t_train_sm)

# 예측 및 리포트
t_pred = model.predict(x_test_scaled)
print(f"\n🔸 모델 성능 🔸")
print(classification_report(t_test, t_pred, digits=3))
# 전체 데이터, 전처리X, 정규화S, f1 39%


🔸 모델 성능 🔸
              precision    recall  f1-score   support

           0      0.931     0.913     0.922     35980
           1      0.334     0.392     0.361      4020

    accuracy                          0.860     40000
   macro avg      0.633     0.653     0.641     40000
weighted avg      0.871     0.860     0.865     40000



In [8]:
# feature 이름이 없으면 var_0 ~ var_199로 생성
feature_names = [f'var_{i}' for i in range(x_train.shape[1])]

# feature importance 추출
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

top_features = [feature_names[i] for i in indices[:100]]
print("상위 100개 변수:", top_features)

상위 100개 변수: ['var_24', 'var_137', 'var_131', 'var_127', 'var_45', 'var_90', 'var_184', 'var_9', 'var_199', 'var_116', 'var_128', 'var_194', 'var_109', 'var_32', 'var_70', 'var_114', 'var_82', 'var_48', 'var_8', 'var_13', 'var_174', 'var_94', 'var_81', 'var_20', 'var_149', 'var_121', 'var_49', 'var_108', 'var_122', 'var_75', 'var_165', 'var_197', 'var_177', 'var_31', 'var_118', 'var_173', 'var_54', 'var_85', 'var_33', 'var_146', 'var_18', 'var_71', 'var_192', 'var_107', 'var_87', 'var_23', 'var_76', 'var_150', 'var_91', 'var_63', 'var_78', 'var_36', 'var_196', 'var_35', 'var_34', 'var_6', 'var_139', 'var_167', 'var_115', 'var_164', 'var_170', 'var_134', 'var_55', 'var_22', 'var_53', 'var_133', 'var_187', 'var_178', 'var_156', 'var_104', 'var_21', 'var_52', 'var_11', 'var_135', 'var_130', 'var_166', 'var_77', 'var_5', 'var_191', 'var_111', 'var_56', 'var_125', 'var_26', 'var_43', 'var_40', 'var_67', 'var_159', 'var_66', 'var_145', 'var_132', 'var_99', 'var_83', 'var_92', 'var_140', 'var_

In [10]:
# 상위 100개 컬럼으로 스타트~

# 1. 상위 100개 변수 인덱스 & 이름 추출
top100_idx = indices[:100]
top100_features = [feature_names[i] for i in top100_idx]

# 2. 원본 데이터에서 해당 feature만 선택
df_top100 = df_1[top100_features + ['target']]  # 'target' 붙여줘야 분리 가능



In [12]:
df_top100.head()

,var_24,var_137,var_131,var_127,var_45,var_90,var_184,var_9,var_199,var_116,...,var_83,var_92,var_140,var_105,var_182,var_169,var_147,var_1,var_12,target
0,14.3831,31.4045,0.3587,-0.7338,-7.0170,-21.4494,25.8398,5.7470,-1.0914,2.5516,...,2.9423,11.0924,8.3307,6.0454,3.0657,5.4879,-16.4727,-6.7863,14.0137,0
1,6.9779,18.1577,-0.1780,2.4354,-47.3797,0.4768,22.5441,8.0851,1.9518,3.0454,...,-4.8210,8.0905,3.6937,2.6227,-14.9100,5.7999,11.7700,-4.1473,14.0239,0
2,5.6777,15.5827,0.0975,-2.5511,-7.9078,-22.4038,23.0866,5.9525,0.3965,1.1696,...,10.1852,14.1613,7.3834,3.9995,-13.2648,5.7690,1.7624,-2.7457,14.1929,0
3,12.1354,24.6065,1.0486,-1.3683,-24.6840,-7.5866,-0.4639,8.2450,-8.9996,5.3446,...,-5.2896,14.4027,3.8873,4.2835,1.8986,5.3430,4.1622,-2.1518,13.8463,0
4,14.2080,25.8128,0.9442,7.0642,-65.4863,-39.7997,11.8503,7.6784,-8.8104,1.4684,...,-6.7075,9.3627,6.6289,5.1934,4.8910,5.5518,-12.7047,-1.4834,13.8481,0


In [13]:
def get_outlier_features(df, threshold_ratio=0.01):
    outlier_info = []

    for col in df.columns:
        if col == 'target':
            continue

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        # 이상치 개수
        outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
        outlier_ratio = outlier_count / len(df)

        if outlier_ratio > threshold_ratio:
            outlier_info.append((col, outlier_count, outlier_ratio))

    # 이상치 비율 순으로 정렬
    outlier_info = sorted(outlier_info, key=lambda x: x[2], reverse=True)

    return pd.DataFrame(outlier_info, columns=['feature', 'outlier_count', 'outlier_ratio'])


In [18]:
outlier_df = get_outlier_features(df_top100, threshold_ratio=0.004)
print(outlier_df.head(20))  # 상위 20개만 보기


   feature  outlier_count  outlier_ratio
0  var_146            804        0.00402


In [ ]:
# 이상치도 저정도면 없는 수준! 패쓰~
# 이제 다시 모델로 돌아와서

In [19]:
# 데이터 분리
xx_data = df_top100.drop('target', axis=1).values
tt_data = df_top100['target'].values

# Train/Test Split
x_data_train, x_data_test, t_data_train, t_data_test = train_test_split(
    xx_data, tt_data,
    test_size=0.2,
    stratify=t_data,
    random_state=42
)

In [20]:
# 스케일링
x_data_train_norm = scaler.fit_transform(x_data_train)
x_data_test_norm = scaler.transform(x_data_test)

# 오버샘플링
x_data_train_sm, t_data_train_sm = SMOTE(random_state=42).fit_resample(x_data_train_norm, t_data_train)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [50, 100],  # 적은 수로 설정
    'max_depth': [3, 4],
    'learning_rate': [0.01, 0.05],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8]
}

xgb = XGBClassifier(use_label_encoder=False, verbosity=0)

rscv = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=params,
    n_iter=30,
    scoring='f1',  # 불균형 데이터일 때 'f1', 'roc_auc' 추천
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rscv.fit(x_data_train_sm, t_data_train_sm)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric=None, feature_types=None,
                                           gamma=None, grow_policy=None,
                                           importance_type=None,
                                           interaction_constraints=None,
                                           learning_rate...
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None,
                                           random_state=None, ...),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.6, 0.8, 1.0],
                                        'gamma': [0, 1, 5],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2],
                                        'max_depth': [3, 4, 5, 6, 7],
                                        'n_estimators': [100, 300, 500, 800],
                                        'reg_alpha': [0, 0.01, 0.1],
                                        'reg_lambda': [0.5, 1.0, 2.0],
                                        'subsample': [0.6, 0.8, 1.0]},
                   random_state=42, scoring='f1', verbose=1)